In [ ]:
# ============================================================
# SERIAL BINARY ADDER - VISUAL STEP-BY-STEP DEMONSTRATION
# ============================================================
#
# This notebook demonstrates how a serial binary adder performs
# the addition of two unsigned binary integers one bit at a time.
#
# HOW TO USE THE NOTEBOOK
#
# 1. Use the "Integer A" and "Integer B" sliders to select the
#    two decimal integers that will be added.
#
# 2. Use the "Word length" slider to select the number of bits
#    used for representing the two input numbers.
#
# 3. Use the "Clock cycle" slider to move through the addition
#    process one bit at a time.
#
# 4. At every clock cycle, the notebook displays:
#
#       - the current bit of A,
#       - the current bit of B,
#       - the carry-in from the previous clock cycle,
#       - the resulting sum bit,
#       - the carry-out sent to the next clock cycle.
#
# 5. The addition proceeds from the least significant bit (LSB)
#    toward the most significant bit (MSB), exactly as in a
#    hardware serial adder.
#
# 6. The partially constructed binary result is shown after
#    every clock cycle.
#
# 7. When the final clock cycle is reached, the complete binary
#    and decimal results are displayed.
#
# 8. Whenever Integer A, Integer B, or the word length changes,
#    a new addition begins and the Clock cycle is automatically
#    reset to 1.
#
# IMPORTANT
#
# The notebook uses unsigned binary integers in order to focus
# exclusively on the operation of the serial adder and on the
# propagation of the carry bit.
#
# ============================================================


from ipywidgets import IntSlider, HBox, VBox, Layout, HTML
from IPython.display import display


# ------------------------------------------------------------
# Global style sheet
# ------------------------------------------------------------

style_html = HTML("""
<style>

.serial-adder-root {
    font-family: monospace;
    width: 100%;
    max-width: 760px;
    box-sizing: border-box;
}

.serial-title {
    font-size: 22px;
    font-weight: bold;
    margin-bottom: 10px;
    color: #1f1f1f;
}

.serial-description {
    font-size: 13px;
    line-height: 1.6;
    padding: 12px 14px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 14px;
    box-sizing: border-box;
    white-space: normal;
}

.section-box {
    border: 1px solid #c8d0dc;
    border-radius: 10px;
    padding: 12px 14px;
    margin-top: 10px;
    background: #ffffff;
    box-sizing: border-box;
    width: 100%;
}

.section-title {
    font-size: 16px;
    font-weight: bold;
    margin-bottom: 10px;
    color: #243447;
}

.info-line {
    font-size: 14px;
    line-height: 1.8;
    white-space: normal;
}

.bit-row {
    margin-top: 6px;
    margin-bottom: 6px;
    white-space: nowrap;
}

.row-label {
    display: inline-block;
    width: 22px;
    font-weight: bold;
    color: #243447;
}

.bit-box {
    display: inline-block;
    width: 29px;
    height: 31px;
    line-height: 31px;
    text-align: center;
    margin-right: 3px;
    border-radius: 5px;
    border: 1px solid #7f8c9a;
    font-size: 16px;
    font-weight: bold;
    box-sizing: border-box;
}

.bit-box-a {
    background: #dceeff;
    color: #0e3a66;
}

.bit-box-b {
    background: #ffe7cf;
    color: #7a3d00;
}

.bit-box-result {
    background: #dff3e4;
    color: #1d5d2d;
}

.bit-box-pending {
    background: #f1f3f5;
    color: #8a8f98;
    border-style: dashed;
}

.bit-box-active {
    border: 3px solid #d62828 !important;
    background: #fff3bf !important;
    color: #111111 !important;
}

.operation-strip {
    display: flex;
    align-items: center;
    gap: 5px;
    flex-wrap: wrap;
    margin-top: 8px;
    margin-bottom: 12px;
}

.op-symbol {
    font-size: 20px;
    font-weight: bold;
    color: #333333;
    padding: 0 1px;
}

.small-caption {
    font-size: 12px;
    line-height: 1.45;
    color: #555555;
    margin-top: 4px;
    margin-bottom: 8px;
    white-space: normal;
}

.kv-table {
    font-size: 14px;
    line-height: 1.9;
}

.kv-key {
    display: inline-block;
    width: 160px;
    font-weight: bold;
    color: #243447;
}

.result-big {
    font-size: 20px;
    font-weight: bold;
    color: #1f3b4d;
    margin-top: 6px;
    margin-bottom: 10px;
}

.note-ok {
    color: #1d5d2d;
    font-weight: bold;
}

.note-wait {
    color: #8a5a00;
    font-weight: bold;
}

.controls-title {
    font-family: monospace;
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 8px;
}

</style>
""")


# ------------------------------------------------------------
# Core arithmetic
# ------------------------------------------------------------

def binary_addition_steps(A, B, bits):
    A_binary = format(A, f'0{bits}b')
    B_binary = format(B, f'0{bits}b')

    carry = 0
    steps = []
    result_bits_lsb = []

    for position in range(bits):
        a_bit = int(A_binary[-1 - position])
        b_bit = int(B_binary[-1 - position])

        total = a_bit + b_bit + carry
        sum_bit = total % 2
        carry_out = total // 2

        result_bits_lsb.append(str(sum_bit))

        steps.append({
            'position': position,
            'a_bit': a_bit,
            'b_bit': b_bit,
            'carry_in': carry,
            'sum_bit': sum_bit,
            'carry_out': carry_out
        })

        carry = carry_out

    if carry == 1:
        final_binary = '1' + ''.join(reversed(result_bits_lsb))
    else:
        final_binary = ''.join(reversed(result_bits_lsb))

    return A_binary, B_binary, steps, final_binary


# ------------------------------------------------------------
# HTML rendering helpers
# ------------------------------------------------------------

def bit_box(bit, css_class):
    return f"<span class='bit-box {css_class}'>{bit}</span>"


def render_binary_row(label, binary_string, current_index, base_class):
    boxes = []

    for i, bit in enumerate(binary_string):
        css = base_class

        if i == current_index:
            css += " bit-box-active"

        boxes.append(bit_box(bit, css))

    return f"""
    <div class="bit-row">
        <span class="row-label">{label}</span>
        {''.join(boxes)}
    </div>
    """


def render_result_row(partial_display, computed_mask):
    boxes = []

    for bit, computed in zip(partial_display, computed_mask):
        if computed:
            boxes.append(bit_box(bit, "bit-box-result"))
        else:
            boxes.append(bit_box(bit, "bit-box-pending"))

    return f"""
    <div class="bit-row">
        <span class="row-label">S</span>
        {''.join(boxes)}
    </div>
    """


# ------------------------------------------------------------
# UI widgets
# ------------------------------------------------------------

title_html = HTML("""
<div class="serial-adder-root">
    <div class="serial-title">
        Serial Binary Adder
    </div>
</div>
""")


description_html = HTML("""
<div class="serial-adder-root">

    <div class="serial-description">

        A serial adder processes one pair of bits during each clock cycle.<br><br>

        The addition starts with the <b>least significant bits (LSB)</b>
        of the two numbers. The carry generated during one clock cycle
        is stored and used as the <b>carry-in</b> of the next clock cycle.<br><br>

        Use the <b>Clock cycle</b> slider to follow the propagation of
        the carry and the progressive construction of the binary result.<br><br>

        Whenever <b>Integer A</b>, <b>Integer B</b>, or the
        <b>Word length</b> changes, the clock cycle automatically
        returns to 1 because a new addition operation starts.

    </div>

</div>
""")


summary_html = HTML()

step_html = HTML()

result_html = HTML()


slider_layout = Layout(width='250px')

style_opts = {'description_width': '88px'}


word_length_slider = IntSlider(
    min=3,
    max=12,
    step=1,
    value=6,
    description='Word length:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


initial_maximum = 2**word_length_slider.value - 1


a_slider = IntSlider(
    min=0,
    max=initial_maximum,
    step=1,
    value=45,
    description='Integer A:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


b_slider = IntSlider(
    min=0,
    max=initial_maximum,
    step=1,
    value=19,
    description='Integer B:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


clock_slider = IntSlider(
    min=1,
    max=word_length_slider.value,
    step=1,
    value=1,
    description='Clock cycle:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


controls_title = HTML("""
<div class="controls-title">
Controls
</div>
""")


# ------------------------------------------------------------
# Dynamic display
# ------------------------------------------------------------

def update_display(*args):
    A = a_slider.value
    B = b_slider.value
    bits = word_length_slider.value
    clock = clock_slider.value

    A_binary, B_binary, steps, final_binary = binary_addition_steps(A, B, bits)

    current_step = steps[clock - 1]

    current_index = bits - clock

    processed_steps = steps[:clock]

    partial_bits = ''.join(str(step['sum_bit']) for step in reversed(processed_steps))

    remaining_bits = bits - clock

    partial_display = ('·' * remaining_bits) + partial_bits

    computed_mask = ([False] * remaining_bits) + ([True] * clock)

    if clock == bits and current_step['carry_out'] == 1:
        partial_display = '1' + partial_display
        computed_mask = [True] + computed_mask


    # --------------------------------------------------------
    # Input section
    # --------------------------------------------------------

    summary_html.value = f"""
    <div class="serial-adder-root">

        <div class="section-box">

            <div class="section-title">
                Input Numbers and Binary Representation
            </div>

            <div class="info-line">
                <b>Decimal values</b><br>
                A = {A}<br>
                B = {B}<br>
                Word length = {bits} bits
            </div>

            <div style="margin-top:10px;">

                {render_binary_row('A', A_binary, current_index, 'bit-box-a')}

                {render_binary_row('B', B_binary, current_index, 'bit-box-b')}

            </div>

            <div class="small-caption">
                The highlighted boxes indicate the pair of bits processed
                during the current clock cycle.
            </div>

        </div>

    </div>
    """


    # --------------------------------------------------------
    # Clock-cycle section
    # --------------------------------------------------------

    total_now = current_step['a_bit'] + current_step['b_bit'] + current_step['carry_in']


    step_html.value = f"""
    <div style="font-family:monospace; width:100%;">

        <div class="section-box">

            <div class="section-title">
                Current Clock Cycle
            </div>

            <div class="info-line">
                <b>Clock cycle:</b> {clock}<br>
                <b>Bit position from LSB:</b> {current_step['position']}
            </div>

            <div class="operation-strip">

                <div style="text-align:center;">
                    {bit_box(current_step['a_bit'], 'bit-box-a')}
                    <div class="small-caption">A bit</div>
                </div>

                <div class="op-symbol">+</div>

                <div style="text-align:center;">
                    {bit_box(current_step['b_bit'], 'bit-box-b')}
                    <div class="small-caption">B bit</div>
                </div>

                <div class="op-symbol">+</div>

                <div style="text-align:center;">
                    {bit_box(current_step['carry_in'], 'bit-box-active')}
                    <div class="small-caption">Carry-in</div>
                </div>

                <div class="op-symbol">=</div>

                <div style="text-align:center;">
                    {bit_box(current_step['sum_bit'], 'bit-box-result')}
                    <div class="small-caption">Sum bit</div>
                </div>

                <div class="op-symbol">,</div>

                <div style="text-align:center;">
                    {bit_box(current_step['carry_out'], 'bit-box-active')}
                    <div class="small-caption">Carry-out</div>
                </div>

            </div>

            <div class="kv-table">

                <span class="kv-key">
                    Current operation
                </span>
                {current_step['a_bit']} + {current_step['b_bit']} +
                {current_step['carry_in']} = {total_now}

                <br>

                <span class="kv-key">
                    Stored sum bit
                </span>
                {current_step['sum_bit']}

                <br>

                <span class="kv-key">
                    Carry sent forward
                </span>
                {current_step['carry_out']}

            </div>

        </div>

    </div>
    """


    # --------------------------------------------------------
    # Result section
    # --------------------------------------------------------

    if clock < bits:

        status_line = """
        <span class="note-wait">
            The addition is still in progress.
        </span>
        """

        final_text = ""

    else:

        status_line = """
        <span class="note-ok">
            The addition is complete.
        </span>
        """

        final_text = f"""
        <div class="info-line" style="margin-top:10px;">

            <b>Complete binary result</b><br>

            <div class="result-big">
                {final_binary}
            </div>

            <b>Decimal result</b><br>

            <div class="result-big">
                {A} + {B} = {A + B}
            </div>

        </div>
        """


    result_html.value = f"""
    <div class="serial-adder-root">

        <div class="section-box">

            <div class="section-title">
                Partial / Final Result
            </div>

            {render_result_row(partial_display, computed_mask)}

            <div class="small-caption">
                Green boxes show already computed bits. Gray dashed boxes
                indicate bit positions that have not been processed yet.
            </div>

            <div class="info-line">

                <b>Current visible result</b><br>

                <div class="result-big">
                    {partial_display}
                </div>

                {status_line}

            </div>

            {final_text}

        </div>

    </div>
    """


# ------------------------------------------------------------
# Reset behavior
# ------------------------------------------------------------

def reset_clock_and_update(change):
    if clock_slider.value != 1:
        clock_slider.value = 1
    else:
        update_display()


def update_word_length(change):
    bits = change['new']

    maximum = 2**bits - 1

    a_slider.max = maximum
    b_slider.max = maximum

    if a_slider.value > maximum:
        a_slider.value = maximum

    if b_slider.value > maximum:
        b_slider.value = maximum

    clock_slider.max = bits

    if clock_slider.value != 1:
        clock_slider.value = 1
    else:
        update_display()


word_length_slider.observe(update_word_length, names='value')

a_slider.observe(reset_clock_and_update, names='value')

b_slider.observe(reset_clock_and_update, names='value')

clock_slider.observe(update_display, names='value')


# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

summary_html.layout = Layout(
    width='760px',
    max_width='760px',
    overflow='visible'
)


step_html.layout = Layout(
    width='460px',
    min_width='460px',
    overflow='visible'
)


controls_box = VBox(
    [
        controls_title,
        a_slider,
        b_slider,
        word_length_slider,
        clock_slider
    ],
    layout=Layout(
        width='280px',
        min_width='280px',
        border='1px solid #c8d0dc',
        padding='12px',
        margin='10px 0 0 10px',
        overflow='visible'
    )
)


middle_row = HBox(
    [
        step_html,
        controls_box
    ],
    layout=Layout(
        width='760px',
        max_width='760px',
        align_items='flex-start',
        justify_content='flex-start',
        overflow='visible'
    )
)


result_html.layout = Layout(
    width='760px',
    max_width='760px',
    overflow='visible'
)


main_layout = VBox(
    [
        summary_html,
        middle_row,
        result_html
    ],
    layout=Layout(
        width='760px',
        max_width='760px',
        align_items='flex-start',
        overflow='visible'
    )
)


# ------------------------------------------------------------
# Initial display
# ------------------------------------------------------------

update_display()

display(style_html)

display(title_html)

display(description_html)

display(main_layout)